# 01 — Phase 1 cycle pretrain + eval + probe

**Paradigm sweet spot setup (revision 4, ph1_v3_minimal)**: batch=32, lr=1e-4, λ_lb=0.1, epoch=30.

Aggressive (batch=256, lr=8e-4, epoch=200) 와 fix (batch=128, λ_lb=0.5) 는 K_active collapse 또는 partial collapse 됨. minimal 만 K_active=13.9 로 K=16 paradigm-aligned.

**Pre-req**: 00_setup 먼저 실행.

In [ ]:
# Pre-setup — Colab kernel 마다 첫 cell. 이미 mount 됐으면 skip.
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

## A. Cycle pretrain (~55분)

Auto-resume 작동 — session timeout 시 같은 cell 재실행.

In [ ]:
from phase1.train import train_phase1

train_phase1(
    run_id='ph1_v3_minimal',
    epochs=30,
    batch_size=32,
    lr=1e-4,
    warmup_steps=300,
    grad_clip=1.0,
    lambda_lb=0.1,
    lambda_ortho=0.0,
)

## B. History trajectory + plateau check

K_active monotonic 증가 + loss_recon plateau 확인.

In [ ]:
import json
from pathlib import Path

run_id = 'ph1_v3_minimal'
h = json.loads(Path(f'out/phase1/{run_id}/history.json').read_text())
first, last = h[0], h[-1]
print(f'[{run_id}] epoch {first["epoch"]} \u2192 {last["epoch"]}')
print(f'  loss        {first["loss"]:.4f} \u2192 {last["loss"]:.4f}')
print(f'  loss_recon  {first["loss_recon"]:.4f} \u2192 {last["loss_recon"]:.4f}')
print(f'  K_active    {first["active_frac"]*16:.2f} \u2192 {last["active_frac"]*16:.2f}  (of 16)')
print(f'  lambda_mean {first["lambda_mean"]:+.3f} \u2192 {last["lambda_mean"]:+.3f}')

if len(h) >= 20:
    last10 = sum(e['loss_recon'] for e in h[-10:]) / 10
    prev10 = sum(e['loss_recon'] for e in h[-20:-10]) / 10
    drop_pct = (prev10 - last10) / prev10 * 100
    print(f'  plateau     last10 recon {last10:.4f}, prev10 {prev10:.4f}, drop {drop_pct:+.2f}%')
    if abs(drop_pct) < 0.1:
        print('              \u2192 PLATEAU REACHED')

## C. Eval — cycle reconstruction + linear probe (~10분)

In [ ]:
from phase1.eval import eval_phase1

result = eval_phase1(run_id='ph1_v3_minimal')
print('\n=== summary ===')
print(f'mean_cosine: {result["cycle_reconstruction"]["mean_cosine"]:.4f}')
print(f'mean_mse:    {result["cycle_reconstruction"]["mean_mse"]:.6f}')
print(f'alpha macro_f1: {result["linear_probe_routed_alpha"].get("macro_f1"):.4f}')
print(f'subkg macro_f1: {result["linear_probe_sub_kg"].get("macro_f1"):.4f}')

## D. Multi-label probe — routing specialization dimension

Source binary 의 imbalance (Pop 97.9% : 2.1%) 가 alpha_F1=0.4942 의 majority predictor baseline 만들음. Balanced + length + author probe 로 진짜 specialization dimension 파악.

**기대치**:
- routed_alpha source_balanced > 0.60 → paradigm specialization 살아있음
- routed_alpha length ≈ random (0.20) → length-agnostic (semantic specialization)
- routed_alpha top-20 author > 0.15 → 자연 발생 user signature (Phase 2 motivation)

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from phase1.eval import linear_probe_separability
from phase1.cycle import CycleConfig, Phase1Cycle, collect_activations
from phase1.data import CorpusConfig, load_phase1_corpus, Phase1Dataset
from phase1.train import MODEL_NAME, CONFIG_NAME

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
run_id = 'ph1_v3_minimal'
run_dir = Path(f'out/phase1/{run_id}')
cfg = json.loads((run_dir / CONFIG_NAME).read_text())

ccfg = CorpusConfig(encoder_name=cfg['encoder_name'], encoder_max_length=cfg['encoder_max_length'])
corpus, fact_emb = load_phase1_corpus(cfg=ccfg)

model = Phase1Cycle(
    n_users=int(corpus['user_id'].nunique()), encoder_name=cfg['encoder_name'],
    k_routed=cfg.get('k_routed', 16), k_shared=cfg.get('k_shared', 4),
    d_hidden=cfg.get('d_hidden', 2048),
    config=CycleConfig(
        lambda_lb=cfg.get('lambda_lb', 0.1),
        lambda_ortho=cfg.get('lambda_ortho', 0.0),
        d_bottleneck=cfg.get('d_bottleneck', 64),
    ),
    use_user=cfg.get('use_user', False),
).to(device)
model.load_state_dict(torch.load(run_dir / MODEL_NAME, map_location=device), strict=False)
model.eval()

test_mask = (corpus['split'] == 'test').values
test_corpus = corpus[test_mask].reset_index(drop=True)
test_ds = Phase1Dataset(fact_emb[test_mask], corpus['user_id'].values[test_mask])
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
acts = collect_activations(model, test_loader, device)
routed_alpha = acts['routed_alpha'].numpy()
sub_kg = acts['sub_kg'].numpy()

def probe(features, labels, name, n_classes):
    r = linear_probe_separability(features, labels)
    if 'error' in r:
        print(f'  {name}: ERROR {r["error"]}'); return
    print(f'  {name:<15}  acc={r["accuracy"]:.4f}  macro_f1={r["macro_f1"]:.4f}  ({n_classes} cls)')

# 1. Source BALANCED
print('=== 1. Source binary BALANCED ===')
penn_idx = test_corpus.index[test_corpus['source'] == 'pennebaker'].to_numpy()
reddit_idx = test_corpus.index[test_corpus['source'] == 'reddit'].to_numpy()
rng = np.random.default_rng(42)
reddit_sample = rng.choice(reddit_idx, size=len(penn_idx), replace=False)
bal_idx = np.concatenate([penn_idx, reddit_sample])
bal_labels = (test_corpus.loc[bal_idx, 'source'].values == 'pennebaker').astype(np.int64)
probe(routed_alpha[bal_idx], bal_labels, 'routed_alpha', 2)
probe(sub_kg[bal_idx], bal_labels, 'sub_kg', 2)

# 2. Length 5-quantile
print('\n=== 2. Text length 5-quantile ===')
text_len = test_corpus['text'].str.len().values
quantiles = np.quantile(text_len, [0.2, 0.4, 0.6, 0.8])
length_class = np.searchsorted(quantiles, text_len)
probe(routed_alpha, length_class, 'routed_alpha', 5)
probe(sub_kg, length_class, 'sub_kg', 5)

# 3. Top-20 most active authors
print('\n=== 3. Top-20 most active authors ===')
TOP_K = 20
user_counts = test_corpus['user_id'].value_counts()
top_users = set(user_counts.head(TOP_K).index)
top_idx = np.where(test_corpus['user_id'].isin(top_users).values)[0]
top_labels = test_corpus.loc[top_idx, 'user_id'].values
print(f'  n={len(top_idx)} samples covering {TOP_K} authors')
probe(routed_alpha[top_idx], top_labels, 'routed_alpha', TOP_K)
probe(sub_kg[top_idx], top_labels, 'sub_kg', TOP_K)